# GeoPulse — LightGBM Market Signal Predictor

**What this notebook does:**
1. Downloads ~11 years of GDELT GKG bulk data (2013–present) and extracts real geopolitical tension features
2. Downloads matching market data from Yahoo Finance + FRED (VIX, gold, oil, DXY, 10Y yield)
3. Engineers a rich feature matrix combining geopolitical signals with market microstructure
4. Trains four LightGBM classifiers (one per target) with Optuna hyperparameter search
5. Evaluates with classification reports, confusion matrices, SHAP, and calibration curves
6. Exports a single `lgbm_predictor.pkl` that the production FastAPI loads directly

**Targets predicted:**
- `vix` — VIX direction next 24h: down / neutral / up
- `gold` — Gold bias next 24h: bearish / neutral / bullish
- `oil` — WTI crude bias next 24h: bearish / neutral / bullish
- `risk` — Macro risk quartile: Q1–Q4

**Runtime:** CPU is fine. Estimated ~25–40 min on Colab free tier (mostly GDELT download).

## 0. Install dependencies

In [ ]:
%%capture
!pip install lightgbm==4.3.0 yfinance pandas-datareader shap optuna scikit-learn matplotlib seaborn requests tqdm pyarrow fastparquet

In [ ]:
import warnings, os, io, zipfile, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import pandas_datareader.data as web
import lightgbm as lgb
import shap
import optuna
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)
from sklearn.calibration import calibration_curve

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED  = 42
np.random.seed(SEED)

# GDELT GKG available from April 2013
START    = '2013-04-01'
END      = datetime.today().strftime('%Y-%m-%d')
HORIZONS = [1, 2, 3]   # trading days (1d=24h, 2d=48h, 3d=72h)

print(f'Training period: {START} -> {END}')
print(f'Approx trading days: {len(pd.bdate_range(START, END))}')

## 1. Download GDELT GKG bulk data

GDELT publishes daily `.csv.zip` GKG files. Each row is one article. We extract:
- `V2Tone` — tone metrics: avg_tone, pos_score, neg_score, polarity, activity_ref_density
- `GCAM` — pre-computed sentiment dimensions (conflict intensity, protest, economic stress)
- `Themes` — pipe-separated themes (we count conflict/geopolitical themes)
- `Locations` — geographic diversity (# unique country codes)

We aggregate to **daily** resolution, then merge with market data.

In [ ]:
# GKG column spec (v2.1)
GKG_COLS = [
    'GKGRECORDID','DATE','SourceCollectionIdentifier','SourceCommonName',
    'DocumentIdentifier','Counts','V2Counts','Themes','V2Themes',
    'Locations','V2Locations','Persons','V2Persons','Organizations',
    'V2Organizations','V2Tone','Dates','GCAM','SharingImage',
    'RelatedImages','SocialImageEmbeds','SocialVideoEmbeds','Quotations',
    'AllNames','Amounts','TranslationInfo','Extras'
]

# GCAM dimension IDs -> human names
# Full codebook: https://www.gdeltproject.org/data/lookups/GCAM-MASTER-CODEBOOK.TXT
GCAM_DIMS = {
    'c18.134': 'gcam_conflict',
    'c18.135': 'gcam_protest',
    'c18.136': 'gcam_econ_stress',
    'c18.22':  'gcam_pol_instability',
    'c18.44':  'gcam_humanitarian',
    'wc':      'gcam_word_count',
}

CONFLICT_THEMES = {
    'CONFLICT','WAR','TERROR','PROTEST','UNREST','SANCTION',
    'MILITARY','WEAPONS','NUCLEAR','HUMANITARIAN_CRISIS',
    'REFUGEE','COUP','OIL','ENERGY','ECONOMIC_CRISIS','RECESSION'
}


def parse_v2tone(s):
    keys = ['tone_avg','tone_pos','tone_neg','tone_polarity','tone_ard','tone_srd']
    try:
        vals = [float(x) for x in str(s).split(',')[:6]]
        return dict(zip(keys, vals + [0.0]*(6-len(vals))))
    except:
        return {k: 0.0 for k in keys}


def parse_gcam(s, dims):
    result = {v: 0.0 for v in dims.values()}
    if not s or pd.isna(s):
        return result
    for item in str(s).split(';'):
        parts = item.split(',')
        if len(parts)==2 and parts[0] in dims:
            try:
                result[dims[parts[0]]] = float(parts[1])
            except:
                pass
    return result


def count_conflict_themes(s):
    if not s or pd.isna(s):
        return 0
    themes = set(str(s).upper().replace(';',',').split(','))
    return sum(1 for t in CONFLICT_THEMES if any(t in th for th in themes))


def count_unique_countries(s):
    if not s or pd.isna(s):
        return 0
    codes = set()
    for loc in str(s).split('#'):
        parts = loc.split(';')
        if len(parts)>=3 and len(parts[2])==2:
            codes.add(parts[2])
    return len(codes)


def fetch_one_day(date):
    url = f"http://data.gdeltproject.org/gkg/{date.strftime('%Y%m%d')}.gkg.csv.zip"
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code != 200:
            return None
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            with zf.open(zf.namelist()[0]) as f:
                df = pd.read_csv(
                    f, sep='\t', header=None, names=GKG_COLS,
                    usecols=['DATE','V2Tone','GCAM','Themes','Locations'],
                    dtype=str, on_bad_lines='skip', low_memory=False
                )
    except Exception:
        return None

    if df.empty:
        return None

    n = len(df)
    tone_df = pd.DataFrame(df['V2Tone'].apply(parse_v2tone).tolist())
    gcam_df = pd.DataFrame(df['GCAM'].apply(lambda x: parse_gcam(x, GCAM_DIMS)).tolist())
    df['n_conflict_themes'] = df['Themes'].apply(count_conflict_themes)
    df['n_countries']       = df['Locations'].apply(count_unique_countries)

    agg = {'date': date.strftime('%Y-%m-%d'), 'n_articles': n}
    agg['n_conflict_articles']    = int((df['n_conflict_themes']>0).sum())
    agg['conflict_article_pct']   = agg['n_conflict_articles'] / max(n,1)
    agg['geo_diversity']          = float(df['n_countries'].mean())
    agg['conflict_theme_density'] = float(df['n_conflict_themes'].mean())

    for col in tone_df.columns:
        agg[f'{col}_mean'] = float(tone_df[col].mean())
        agg[f'{col}_std']  = float(tone_df[col].std())
    for col in gcam_df.columns:
        agg[f'{col}_mean'] = float(gcam_df[col].mean())
        agg[f'{col}_p90']  = float(gcam_df[col].quantile(0.90))

    return pd.DataFrame([agg])


print('GDELT fetch functions defined')
print('Testing with a recent date...')
_t = fetch_one_day(datetime.today()-timedelta(days=3))
if _t is not None:
    print(f'  OK - {len(_t.columns)} features, n_articles={_t["n_articles"].iloc[0]}')
else:
    print('  Fetch failed for test date - will retry in the main loop')

In [ ]:
# Main GDELT download loop
# Checkpoints every 100 days so you can resume if Colab disconnects.
# Estimated time: 30-45 min (network bound, ~3,000 files at ~100KB each)

CHECKPOINT = 'gdelt_checkpoint.parquet'
OUTPUT     = 'gdelt_daily.parquet'

all_dates = pd.bdate_range(start=START, end=END)
print(f'Business days to fetch: {len(all_dates)}')

if os.path.exists(CHECKPOINT):
    existing = pd.read_parquet(CHECKPOINT)
    done = set(pd.to_datetime(existing['date']).dt.strftime('%Y-%m-%d'))
    todo = [d for d in all_dates if d.strftime('%Y-%m-%d') not in done]
    frames = [existing]
    print(f'Resuming: {len(done)} done, {len(todo)} remaining')
else:
    todo   = list(all_dates)
    frames = []
    print('Starting fresh')

errors = 0
for i, date in enumerate(tqdm(todo, desc='GDELT GKG')):
    r = fetch_one_day(date)
    if r is not None:
        frames.append(r)
    else:
        errors += 1
    if frames and i % 100 == 99:
        pd.concat(frames, ignore_index=True).to_parquet(CHECKPOINT, index=False)

gdelt_raw = pd.concat(frames, ignore_index=True)
gdelt_raw['date'] = pd.to_datetime(gdelt_raw['date'])
gdelt_raw = gdelt_raw.sort_values('date').drop_duplicates('date').reset_index(drop=True)
gdelt_raw.to_parquet(OUTPUT, index=False)

print(f'Done. {len(gdelt_raw)} days, {errors} errors')
print(f'Range: {gdelt_raw["date"].min().date()} -> {gdelt_raw["date"].max().date()}')
gdelt_raw.head(3)

## 2. Download market data

- **Yahoo Finance**: VIX, Gold, WTI Oil, Brent, Wheat, DXY, 10Y yield, S&P 500
- **FRED**: Fed Funds Rate, CPI, 10Y-2Y yield curve, HY credit spread

In [ ]:
YAHOO_TICKERS = {
    '^VIX':     'vix',
    'GC=F':     'gold',
    'CL=F':     'oil_wti',
    'BZ=F':     'oil_brent',
    'ZW=F':     'wheat',
    'DX-Y.NYB': 'dxy',
    '^TNX':     'us10y',
    'SPY':      'sp500',
}

FRED_SERIES = {
    'FEDFUNDS':      'fed_funds',
    'CPIAUCSL':      'cpi',
    'T10Y2Y':        'yield_curve',
    'BAMLH0A0HYM2':  'hy_spread',
}

print('Downloading Yahoo Finance...')
yf_dfs = []
for ticker, name in YAHOO_TICKERS.items():
    try:
        d = yf.download(ticker, start=START, end=END, interval='1d',
                        progress=False, auto_adjust=True)
        if not d.empty:
            parts = [d['Close'].rename(name), d['High'].rename(f'{name}_high'),
                     d['Low'].rename(f'{name}_low')]
            if 'Volume' in d:
                parts.append(d['Volume'].rename(f'{name}_vol'))
            yf_dfs.append(pd.concat(parts, axis=1))
            print(f'  {name}: {len(d)} rows')
    except Exception as e:
        print(f'  {name}: {e}')

market_yf = pd.concat(yf_dfs, axis=1)
market_yf.index = pd.to_datetime(market_yf.index).normalize()

print('\nDownloading FRED...')
fred_dfs = []
for sid, name in FRED_SERIES.items():
    try:
        d = web.DataReader(sid, 'fred', start=START, end=END)
        fred_dfs.append(d.rename(columns={sid: name})[name])
        print(f'  {name}: {len(d)} rows')
    except Exception as e:
        print(f'  {name}: {e}')

if fred_dfs:
    market_fred = pd.concat(fred_dfs, axis=1)
    market_fred.index = pd.to_datetime(market_fred.index).normalize()
    market_fred = market_fred.resample('B').last().ffill()
    market = market_yf.join(market_fred, how='left').ffill()
else:
    market = market_yf.copy()

market = market.resample('B').last().ffill().bfill()
print(f'\nCombined market data: {market.shape}')

## 3. Feature engineering

**A. GDELT features** — raw, rolling windows (3/7/14/30d), z-scores, momentum

**B. Market microstructure** — returns at lags (1/2/3/5/10/21d), realised vol, VIX regime, yield curve

**C. Interaction features** — tension × VIX, conflict × oil, negativity × gold, calendar effects

In [ ]:
def make_gdelt_features(gdf):
    g = gdf.set_index('date').copy()
    g.index = pd.to_datetime(g.index)
    g = g.resample('B').last().ffill()

    raw_cols = [
        c for c in [
            'tone_avg_mean','tone_neg_mean','tone_pos_mean','tone_polarity_mean',
            'gcam_conflict_mean','gcam_conflict_p90',
            'gcam_protest_mean','gcam_econ_stress_mean',
            'gcam_pol_instability_mean','gcam_humanitarian_mean',
            'n_articles','n_conflict_articles','conflict_article_pct',
            'conflict_theme_density','geo_diversity',
        ] if c in g.columns
    ]

    feat = g[raw_cols].copy()
    for col in raw_cols:
        for w in [3,7,14,30]:
            feat[f'{col}_ma{w}']  = g[col].rolling(w).mean()
            feat[f'{col}_std{w}'] = g[col].rolling(w).std()
        roll_m = g[col].rolling(30).mean()
        roll_s = g[col].rolling(30).std().replace(0, np.nan)
        feat[f'{col}_z30'] = (g[col] - roll_m) / roll_s
        feat[f'{col}_d1']  = g[col].diff(1)
        feat[f'{col}_d7']  = g[col].diff(7)

    # Composite tension index (mirrors production formula)
    neg  = feat.get('tone_neg_mean', pd.Series(0,index=feat.index))
    conf = feat.get('gcam_conflict_mean', pd.Series(0,index=feat.index))
    prot = feat.get('gcam_protest_mean', pd.Series(0,index=feat.index))
    econ = feat.get('gcam_econ_stress_mean', pd.Series(0,index=feat.index))
    vol  = feat.get('conflict_article_pct', pd.Series(0,index=feat.index))
    raw_t = (0.30*neg.clip(0,20)/20 + 0.25*conf.clip(0) +
              0.15*prot.clip(0) + 0.15*econ.clip(0) + 0.15*vol) * 100
    feat['tension_index']  = raw_t.clip(0,100)
    feat['tension_ma7']    = feat['tension_index'].rolling(7).mean()
    feat['tension_ma30']   = feat['tension_index'].rolling(30).mean()
    feat['tension_d1']     = feat['tension_index'].diff(1)
    feat['tension_d7']     = feat['tension_index'].diff(7)
    feat['tension_z30']    = (
        (feat['tension_index'] - feat['tension_index'].rolling(30).mean()) /
        feat['tension_index'].rolling(30).std().replace(0,np.nan)
    )
    streak = (feat['tension_index'] > 60).astype(int)
    feat['tension_streak'] = streak.groupby((streak==0).cumsum()).cumcount()
    return feat


def make_market_features(mdf):
    feat = pd.DataFrame(index=mdf.index)
    price_cols = [c for c in ['vix','gold','oil_wti','oil_brent','wheat','dxy','us10y','sp500'] if c in mdf.columns]
    for col in price_cols:
        s   = mdf[col]
        ret = s.pct_change() * 100
        feat[f'{col}_level'] = s
        feat[f'{col}_ret1']  = ret
        for lag in [1,2,3,5,10,21]:
            feat[f'{col}_ret{lag}d'] = s.pct_change(lag)*100
            feat[f'{col}_lag{lag}']  = s.shift(lag)
        for w in [5,10,21]:
            feat[f'{col}_rvol{w}'] = ret.rolling(w).std()
        if f'{col}_high' in mdf.columns and f'{col}_low' in mdf.columns:
            feat[f'{col}_range'] = (mdf[f'{col}_high']-mdf[f'{col}_low'])/mdf[col]
    if 'yield_curve' in mdf.columns:
        feat['yield_curve'] = mdf['yield_curve']
        feat['yield_inv']   = (mdf['yield_curve']<0).astype(int)
    if 'hy_spread' in mdf.columns:
        feat['hy_spread']      = mdf['hy_spread']
        feat['hy_spread_ma21'] = mdf['hy_spread'].rolling(21).mean()
        feat['hy_spread_z30']  = ((mdf['hy_spread']-mdf['hy_spread'].rolling(30).mean())
                                   /mdf['hy_spread'].rolling(30).std().replace(0,np.nan))
    if 'fed_funds' in mdf.columns:
        feat['fed_funds']    = mdf['fed_funds']
        feat['fed_funds_d1'] = mdf['fed_funds'].diff(1)
    if 'vix' in mdf.columns:
        feat['vix_above_20'] = (mdf['vix']>20).astype(int)
        feat['vix_above_30'] = (mdf['vix']>30).astype(int)
        feat['vix_ma21']     = mdf['vix'].rolling(21).mean()
        feat['vix_zscore']   = ((mdf['vix']-mdf['vix'].rolling(252).mean())
                                 /mdf['vix'].rolling(252).std().replace(0,np.nan))
    feat['dow']         = feat.index.dayofweek
    feat['month']       = feat.index.month
    feat['quarter_end'] = feat.index.is_quarter_end.astype(int)
    feat['month_end']   = feat.index.is_month_end.astype(int)
    return feat


def make_interaction_features(geo, mkt):
    feat = pd.DataFrame(index=geo.index)
    if 'tension_index' in geo.columns and 'vix_level' in mkt.columns:
        feat['tension_x_vix']  = geo['tension_index'] * mkt['vix_level'] / 100
        feat['tension_x_vixz'] = geo['tension_z30'] * mkt.get('vix_zscore', pd.Series(0,index=mkt.index))
    if 'gcam_conflict_mean' in geo.columns and 'oil_wti_ret1' in mkt.columns:
        feat['conflict_x_oil'] = geo['gcam_conflict_mean'] * mkt['oil_wti_ret1'].abs()
    if 'tone_neg_mean' in geo.columns and 'gold_ret1' in mkt.columns:
        feat['neg_x_gold']     = geo['tone_neg_mean'] * mkt['gold_ret1']
    if 'tension_d7' in geo.columns and 'hy_spread_z30' in mkt.columns:
        feat['tension_roc_x_credit'] = geo['tension_d7'] * mkt['hy_spread_z30']
    if 'tension_streak' in geo.columns:
        feat['tension_streak'] = geo['tension_streak']
    return feat


print('Building features...')
geo_feats = make_gdelt_features(gdelt_raw)
mkt_feats = make_market_features(market)

common = geo_feats.index.intersection(mkt_feats.index)
geo_feats = geo_feats.loc[common]
mkt_feats = mkt_feats.loc[common]
inter_feats = make_interaction_features(geo_feats, mkt_feats)

X_all = pd.concat([geo_feats, mkt_feats, inter_feats], axis=1).loc[common]

# Drop >40% NaN columns, then fill remaining
drop = X_all.columns[X_all.isna().mean()>0.40]
X_all = X_all.drop(columns=drop).ffill().bfill()
print(f'Feature matrix: {X_all.shape} (dropped {len(drop)} high-NaN cols)')
print(f'Remaining NaN: {X_all.isna().sum().sum()}')

## 4. Build target variables

In [ ]:
TARGETS = {'vix':'y_vix', 'gold':'y_gold', 'oil':'y_oil', 'risk':'y_risk'}

TARGET_BINS = {
    'vix':  ([-np.inf,-5.,5.,np.inf],  [0,1,2]),
    'gold': ([-np.inf,-0.5,0.5,np.inf],[0,1,2]),
    'oil':  ([-np.inf,-1.5,1.5,np.inf],[0,1,2]),
    'risk': ([0,15,20,28,np.inf],       [1,2,3,4]),
}

TARGET_LABELS = {
    'vix':  {0:'Down',1:'Neutral',2:'Up'},
    'gold': {0:'Bearish',1:'Neutral',2:'Bullish'},
    'oil':  {0:'Bearish',1:'Neutral',2:'Bullish'},
    'risk': {1:'Q1',2:'Q2',3:'Q3',4:'Q4'},
}

def build_targets(mdf, idx, horizon=1):
    m = mdf.reindex(idx).ffill()
    T = pd.DataFrame(index=idx)
    col_map = {'vix':'vix','gold':'gold','oil':'oil_wti'}
    for name, col in col_map.items():
        if col not in m.columns: continue
        fwd = m[col].shift(-horizon)
        if name == 'risk':
            T['y_risk'] = pd.cut(fwd, bins=TARGET_BINS['risk'][0],
                                  labels=TARGET_BINS['risk'][1]).astype('Int64').astype(float)
        else:
            ret = (fwd - m[col]) / m[col] * 100
            bins, labels = TARGET_BINS[name]
            T[f'y_{name}'] = pd.cut(ret, bins=bins, labels=labels).astype('Int64').astype(float)
    # risk needs vix
    if 'vix' in m.columns:
        fwd_vix = m['vix'].shift(-horizon)
        T['y_risk'] = pd.cut(fwd_vix, bins=TARGET_BINS['risk'][0],
                              labels=TARGET_BINS['risk'][1]).astype('Int64').astype(float)
    return T

all_targets = {}
for h in HORIZONS:
    all_targets[h] = build_targets(market.reindex(X_all.index, method='ffill'), X_all.index, h)
    print(f'Horizon {h}d targets built')

# Plot distributions
fig, axes = plt.subplots(1,4,figsize=(16,3))
T1 = all_targets[1].dropna()
for ax, col in zip(axes, ['y_vix','y_gold','y_oil','y_risk']):
    if col not in T1.columns: continue
    name = col[2:]
    vc = T1[col].value_counts().sort_index()
    vc.index = [TARGET_LABELS[name].get(int(i), str(i)) for i in vc.index]
    vc.plot.bar(ax=ax, color='#1D9E75', edgecolor='none')
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Target distributions (1d horizon)', y=1.02)
plt.tight_layout()
plt.savefig('target_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Hyperparameter optimisation with Optuna

Optuna with `TimeSeriesSplit` — strictly no leakage (validation is always in the future relative to training).

In [ ]:
N_CV     = 5
N_TRIALS = 40   # reduce to 20 for faster runs
HORIZON  = 1    # primary horizon for HPO

tscv = TimeSeriesSplit(n_splits=N_CV)


def make_objective(X, y, n_classes):
    def objective(trial):
        p = dict(
            n_estimators     = trial.suggest_int('n_estimators', 200, 800),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            num_leaves       = trial.suggest_int('num_leaves', 20, 80),
            min_child_samples= trial.suggest_int('min_child_samples', 10, 50),
            feature_fraction = trial.suggest_float('feature_fraction', 0.5, 1.0),
            bagging_fraction = trial.suggest_float('bagging_fraction', 0.5, 1.0),
            bagging_freq     = trial.suggest_int('bagging_freq', 1, 7),
            reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 2.0, log=True),
        )
        scores = []
        for tr, va in tscv.split(X):
            m = lgb.LGBMClassifier(**p, objective='multiclass', num_class=n_classes,
                                    class_weight='balanced', random_state=SEED,
                                    n_jobs=-1, verbose=-1)
            m.fit(X[tr], y[tr], eval_set=[(X[va],y[va])],
                  callbacks=[lgb.early_stopping(30,verbose=False)])
            scores.append(f1_score(y[va], m.predict(X[va]), average='weighted'))
        return float(np.mean(scores))
    return objective


best_params = {}
best_scores = {}

for name, tcol in TARGETS.items():
    if tcol not in all_targets[HORIZON].columns:
        continue
    print(f'\n[{name}] Optimising ({N_TRIALS} trials)...')
    comb = X_all.join(all_targets[HORIZON][[tcol]],how='inner').dropna()
    fcols = [c for c in comb.columns if c!=tcol]
    Xn = comb[fcols].values.astype(np.float32)
    yn = comb[tcol].values.astype(int)
    ulabels = np.unique(yn)
    remap = {old:new for new,old in enumerate(ulabels)}
    yn = np.array([remap[v] for v in yn])
    nc = len(ulabels)

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(make_objective(Xn, yn, nc), n_trials=N_TRIALS, show_progress_bar=True)
    best_params[name] = study.best_params
    best_scores[name] = study.best_value
    print(f'  Best CV F1: {study.best_value:.4f}')

print('\nOptimisation complete:')
for k,v in best_scores.items():
    print(f'  {k}: {v:.4f}')

## 6. Train final models + evaluate

In [ ]:
final_models   = {}
feature_names  = {}
label_remaps   = {}
label_unremaps = {}
eval_results   = {}

for h in HORIZONS:
    final_models[h] = {}
    for name, tcol in TARGETS.items():
        if tcol not in all_targets[h].columns:
            continue
        comb = X_all.join(all_targets[h][[tcol]],how='inner').dropna()
        fcols = [c for c in comb.columns if c!=tcol]
        Xn = comb[fcols].values.astype(np.float32)
        yn = comb[tcol].values.astype(int)
        ulabels = np.unique(yn)
        remap   = {old:new for new,old in enumerate(ulabels)}
        unremap = {new:old for old,new in remap.items()}
        ym = np.array([remap[v] for v in yn])
        nc = len(ulabels)

        params = best_params.get(name, dict(
            n_estimators=400, learning_rate=0.05, num_leaves=40,
            min_child_samples=20, feature_fraction=0.8,
            bagging_fraction=0.8, bagging_freq=3,
            reg_alpha=0.1, reg_lambda=0.1,
        ))

        # Evaluate on last held-out fold
        splits = list(tscv.split(Xn))
        tr_idx, te_idx = splits[-1]
        mev = lgb.LGBMClassifier(**params, objective='multiclass', num_class=nc,
                                   class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)
        mev.fit(Xn[tr_idx], ym[tr_idx])
        yp = mev.predict(Xn[te_idx])
        eval_results[f'{name}_h{h}'] = dict(
            y_true=ym[te_idx], y_pred=yp,
            y_proba=mev.predict_proba(Xn[te_idx]),
            acc=accuracy_score(ym[te_idx],yp),
            f1=f1_score(ym[te_idx],yp,average='weighted'),
            n_classes=nc, unique_labels=ulabels
        )

        # Final model on all data
        mfinal = lgb.LGBMClassifier(**params, objective='multiclass', num_class=nc,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)
        mfinal.fit(Xn, ym)
        final_models[h][name] = mfinal

        if h==1:
            feature_names[name]  = fcols
            label_remaps[name]   = remap
            label_unremaps[name] = unremap

        r = eval_results[f'{name}_h{h}']
        print(f'[h={h}d] {name:5s}  acc={r["acc"]:.3f}  f1={r["f1"]:.3f}  (n_test={len(te_idx)})')

In [ ]:
# Classification reports
for name in TARGETS:
    r = eval_results.get(f'{name}_h1')
    if not r: continue
    n  = r['n_classes']
    nl = list(TARGET_LABELS[name].values())[:n]
    print(f'\n--- {name.upper()} ---')
    print(classification_report(r['y_true'], r['y_pred'], target_names=nl, digits=3))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1,4,figsize=(20,4))
for ax, name in zip(axes, TARGETS):
    r = eval_results.get(f'{name}_h1')
    if not r: ax.set_visible(False); continue
    n  = r['n_classes']
    nl = list(TARGET_LABELS[name].values())[:n]
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=nl, yticklabels=nl)
    ax.set_title(f'{name}  acc={r["acc"]:.3f}', fontsize=10)
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.suptitle('Confusion matrices — 1d horizon, held-out test fold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Accuracy vs horizon
fig, ax = plt.subplots(figsize=(8,4))
cols = {'vix':'#E24B4A','gold':'#EF9F27','oil':'#378ADD','risk':'#7F77DD'}
for name in TARGETS:
    accs = [eval_results.get(f'{name}_h{h}',{}).get('acc',np.nan) for h in HORIZONS]
    ax.plot([f'{h}d' for h in HORIZONS], accs, marker='o', label=name, color=cols[name])
ax.axhline(0.33, linestyle='--', color='gray', alpha=0.5, label='random baseline')
ax.set_title('Accuracy vs prediction horizon')
ax.set_ylabel('Accuracy'); ax.set_xlabel('Horizon')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('accuracy_vs_horizon.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. SHAP feature importance

SHAP tells us which features actually drive each prediction and in which direction. The green bars are geopolitical (GDELT) features — these are the signal that justifies the whole project.

In [ ]:
# SHAP for VIX Up prediction
name  = 'vix'
tcol  = TARGETS[name]
comb  = X_all.join(all_targets[1][[tcol]],how='inner').dropna()
fcols = [c for c in comb.columns if c!=tcol]
Xs    = comb[fcols].values.astype(np.float32)

model_shap = final_models[1][name]
print('Computing SHAP values (~2 min)...')
explainer  = shap.TreeExplainer(model_shap)
idx_s = np.random.choice(len(Xs), min(2000,len(Xs)), replace=False)
svs   = explainer.shap_values(Xs[idx_s])   # list[n_classes] of (n_samples, n_features)

# Class 2 = VIX Up
sv_up = svs[2] if isinstance(svs,list) else svs
plt.figure(figsize=(10,8))
shap.summary_plot(sv_up, Xs[idx_s], feature_names=fcols, max_display=25, show=False)
plt.title('SHAP — VIX Up prediction (top 25 features)')
plt.tight_layout()
plt.savefig('shap_vix_up.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance — all targets, geopolitical features highlighted
geo_kw = ['tension','gcam','tone','conflict','protest','article','theme','geo','gdelt']
fig, axes = plt.subplots(1,4,figsize=(20,7))
for ax, name in zip(axes, TARGETS):
    m = final_models[1].get(name)
    if not m: continue
    fc = feature_names.get(name,[])
    imp = pd.Series(m.feature_importances_, index=fc).sort_values(ascending=False).head(20)
    colors = ['#1D9E75' if any(k in c for k in geo_kw) else '#888780' for c in imp.index]
    imp.plot.barh(ax=ax, color=colors)
    ax.set_title(f'{name} — top 20', fontsize=10)
    ax.set_xlabel('importance')
    ax.invert_yaxis()

from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor='#1D9E75',label='Geopolitical (GDELT)'),
                     Patch(facecolor='#888780',label='Market / macro')],
            loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.06))
plt.suptitle('Feature importance — green = geopolitical signals', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Calibration curves

Checks that confidence scores are meaningful — 70% confidence should correspond to ~70% accuracy.

In [ ]:
fig, axes = plt.subplots(1,4,figsize=(18,4))
for ax, name in zip(axes, TARGETS):
    r = eval_results.get(f'{name}_h1')
    if not r: continue
    ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Perfect')
    nl = list(TARGET_LABELS[name].values())[:r['n_classes']]
    for ci in range(r['n_classes']):
        yb = (r['y_true']==ci).astype(int)
        pr = r['y_proba'][:,ci]
        try:
            pt, pp = calibration_curve(yb, pr, n_bins=10)
            ax.plot(pp, pt, marker='o', markersize=4, label=nl[ci])
        except: pass
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Fraction positive')
    ax.legend(fontsize=7); ax.grid(alpha=0.2)
plt.suptitle('Calibration curves — 1d horizon')
plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Tension vs VIX — geopolitical event case studies

In [ ]:
EVENTS = [
    ('2014-03-18','Russia annexes Crimea'),
    ('2016-06-24','Brexit vote'),
    ('2018-08-10','Turkey currency crisis'),
    ('2019-09-14','Saudi Aramco drone attack'),
    ('2020-03-11','COVID pandemic declared'),
    ('2022-02-24','Russia invades Ukraine'),
    ('2023-10-07','Hamas attack on Israel'),
    ('2024-04-01','Iran attacks Israel'),
]
avail = [(d,l) for d,l in EVENTS
         if pd.Timestamp(d)>=pd.Timestamp(START) and pd.Timestamp(d)<=pd.Timestamp(END)]

tension = X_all['tension_index'].dropna() if 'tension_index' in X_all.columns else None
vix     = market['vix'].reindex(X_all.index).ffill() if 'vix' in market.columns else None

if tension is not None and vix is not None:
    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(16,8),sharex=True)
    ax1.fill_between(tension.index, tension, alpha=0.35, color='#E24B4A')
    ax1.plot(tension.index, tension, color='#E24B4A', lw=0.8)
    ax1.axhline(60, ls='--', color='#E24B4A', alpha=0.5, lw=0.8)
    ax1.set_ylabel('GeoPulse Tension Index')
    ax1.set_title('Geopolitical Tension Index vs VIX with annotated events', fontsize=12)
    ax1.grid(alpha=0.15)
    ax2.fill_between(vix.index, vix, alpha=0.35, color='#378ADD')
    ax2.plot(vix.index, vix, color='#378ADD', lw=0.8)
    ax2.axhline(20, ls='--', color='#378ADD', alpha=0.5, lw=0.8)
    ax2.set_ylabel('VIX'); ax2.grid(alpha=0.15)
    for ds,label in avail:
        ts = pd.Timestamp(ds)
        for ax in [ax1,ax2]:
            ax.axvline(ts, color='#EF9F27', alpha=0.7, lw=1, ls=':')
        ax1.text(ts, ax1.get_ylim()[1]*0.92, label,
                  rotation=90, fontsize=7, color='#EF9F27', va='top', ha='right')
    plt.tight_layout()
    plt.savefig('tension_vs_vix.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('tension_index or vix not available')

## 10. Save model package

In [ ]:
# Package for production API
# The API uses horizon_hours (24/48/72), notebook uses trading days (1/2/3)
model_package = {
    1: final_models.get(1,{}),
    2: final_models.get(2,{}),
    3: final_models.get(3,{}),
    '__meta__': {
        'feature_names':      feature_names,
        'label_remaps':       label_remaps,
        'label_unremaps':     label_unremaps,
        'horizon_map':        {24:1, 48:2, 72:3},
        'targets':            list(TARGETS.keys()),
        'trained_at':         datetime.today().isoformat(),
        'train_start':        START,
        'train_end':          END,
        'cv_f1_scores':       best_scores,
        'test_fold_accuracy': {n: eval_results.get(f'{n}_h1',{}).get('acc') for n in TARGETS},
        'test_fold_f1':       {n: eval_results.get(f'{n}_h1',{}).get('f1')  for n in TARGETS},
    }
}

MODEL_PATH = 'lgbm_predictor.pkl'
with open(MODEL_PATH,'wb') as f:
    pickle.dump(model_package, f, protocol=4)

size_mb = os.path.getsize(MODEL_PATH)/1e6
print(f'Saved: {MODEL_PATH}  ({size_mb:.1f} MB)')

meta = model_package['__meta__']
print(f"\nTrained {meta['train_start']} -> {meta['train_end']}")
print('CV F1 (Optuna best):')
for k,v in meta['cv_f1_scores'].items(): print(f'  {k}: {v:.4f}')
print('Test fold accuracy:')
for k,v in meta['test_fold_accuracy'].items():
    if v: print(f'  {k}: {v:.4f}')

In [ ]:
# Download outputs
all_files = [
    ('lgbm_predictor.pkl',       'Place in geopulse/data/models/'),
    ('gdelt_daily.parquet',      'Keep for retraining'),
    ('tension_vs_vix.png',       'Portfolio README'),
    ('shap_vix_up.png',          'Portfolio README'),
    ('feature_importance.png',   'Portfolio README'),
    ('confusion_matrices.png',   'Evaluation'),
    ('accuracy_vs_horizon.png',  'Evaluation'),
    ('calibration_curves.png',   'Evaluation'),
    ('target_distributions.png', 'Sanity check'),
]
print('Files produced:')
for fname, note in all_files:
    if os.path.exists(fname):
        kb = os.path.getsize(fname)/1e3
        print(f'  {fname:40s} ({kb:6.0f} KB)  <- {note}')

print('\nNext steps:')
print('  1. Download lgbm_predictor.pkl')
print('  2. mv lgbm_predictor.pkl geopulse/data/models/')
print('  3. Restart the FastAPI backend — it loads the model automatically')
print('  4. Hit GET /api/v1/signals/latest to see live predictions')

try:
    from google.colab import files
    files.download('lgbm_predictor.pkl')
    files.download('tension_vs_vix.png')
    files.download('shap_vix_up.png')
except ImportError:
    print('\n(Not in Colab — files are in the current directory)')

## Appendix — upgrading to real DB features

Once your GeoPulse backend has been running for a few weeks and has collected real `region_signals` rows, replace the GDELT download section with:

```python
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(DATABASE_URL_SYNC)

db_tension = pd.read_sql("""
    SELECT
        DATE_TRUNC('day', timestamp)            AS date,
        AVG(tension_index)                       AS tension_mean,
        MAX(tension_index)                       AS tension_max,
        STDDEV(tension_index)                    AS tension_std,
        COUNT(*) FILTER (WHERE tension_index>60) AS n_high_tension,
        AVG(conflict_score)                      AS gcam_conflict_mean,
        AVG(sanctions_score)                     AS sanctions_score,
        AVG(economic_stress_score)               AS gcam_econ_stress_mean,
        SUM(article_count)                       AS n_articles
    FROM region_signals
    GROUP BY DATE_TRUNC('day', timestamp)
    ORDER BY date
""", engine)
db_tension['date'] = pd.to_datetime(db_tension['date'])
```

Then swap `db_tension` in place of the GDELT-derived features. Everything else — feature engineering, model training, export — stays identical. This is the upgrade path: **GDELT for bootstrapping → real DB signals for production retraining**.